# QLoRA Analytics Agent — Google Colab Training

Fine-tune **Qwen2.5-3B-Instruct** with QLoRA for enterprise analytics skill routing + safe SQL generation.

**Runtime:** set to **GPU** (T4 or A100). A 2–3 epoch run is a few dollars of compute units — well under $100.

Steps: install → get data → train → evaluate → download adapters.

## 1. Install dependencies

In [ ]:
!pip -q install "transformers>=4.44" "peft>=0.12" "trl>=0.9.6" "bitsandbytes>=0.43" \
    "accelerate>=0.33" "datasets>=2.20" duckdb sqlglot faker scikit-learn matplotlib pyyaml
import torch; print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 2. Get the project + generate data

Upload the `qlora-analytics-agent/` folder (or clone your repo), then generate the synthetic dataset. All data is synthetic (GlobalTrade Analytics).

In [ ]:
import os
# Option A: clone your repo
# !git clone https://github.com/<you>/qlora-analytics-agent.git
# Option B: upload a zip via the Files panel, then:
# !unzip -q qlora-analytics-agent.zip
os.chdir('qlora-analytics-agent')
!python scripts/01_generate_synthetic_db.py
!python scripts/02_generate_instruction_data.py
!python scripts/03_preprocess.py

## 3. Train QLoRA (Colab config, auto-adaptive batch size)

In [ ]:
# Smoke test first (optional): !python scripts/04_train_qlora.py --smoke
!python scripts/04_train_qlora.py --config configs/train_qlora_colab.yaml --lora_r 16

## 4. (Optional) Rank ablations

In [ ]:
!python scripts/04_train_qlora.py --config configs/train_qlora_colab.yaml --lora_r 8
!python scripts/04_train_qlora.py --config configs/train_qlora_colab.yaml --lora_r 32

## 5. Evaluate + plot

In [ ]:
!python scripts/05_evaluate.py
!python scripts/06_plot_results.py
import json; print(json.dumps(json.load(open('results/metrics.json')), indent=2))

## 6. Download adapters + results

In [ ]:
!zip -qr adapters_and_results.zip results figures
from google.colab import files
files.download('adapters_and_results.zip')